# Sailing Race Outcome Prediction (Random Forest)

## Model Overview

This project trains a Random Forest regression model to predict a sailor’s finishing position (`Score`)
using historical performance aggregates and contextual race features.

A tree-based model is used to capture nonlinear effects from factors like partner pairing and venue.
Lower `Score` values indicate better finishes (1st place is best).

In [40]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [42]:
df = pd.read_csv("races.csv")

# Create sailor-level performance + experience aggregates for modeling
historical_stats = df.groupby('Sailor').agg({
    'Score': ['mean', 'min', 'max', 'count']
}).reset_index()

historical_stats.columns = ['Sailor', 'avg_score', 'best_score', 'worst_score', 'races_completed']

historical_stats.head()

,Sailor,avg_score,best_score,worst_score,races_completed
0,A.J. Crane,11.750000,8,15,4
1,AJ Kozaritz,11.500000,8,15,2
2,AUSTIN SJAARDA,8.076923,2,12,26
3,AXCELLE BELL,4.333333,1,11,12
4,Aaron Blust,11.961538,1,20,52


In [43]:
# Encode categorical division/position fields so the model can use them
le_div = LabelEncoder()
le_position = LabelEncoder()
df['Div_encoded'] = le_div.fit_transform(df['Div'])
df['Position_encoded'] = le_position.fit_transform(df['Position'])

## Partner and Venue Features

`Partner` and `Venue` have many unique values, so one-hot encoding would add a large number
of mostly empty columns to the dataset.
Instead, I summarize each using average finish position and race count,
which captures how strong a partner or venue tends to be without adding unnecessary complexity.

In [47]:
# Build partner/venue aggregate features and merge into a single modeling table.
partner_stats = df.groupby('Partner')['Score'].agg(['mean', 'count']).reset_index()
partner_stats.columns = ['Partner', 'partner_avg_score', 'partner_races']

venue_stats = df.groupby('Venue')['Score'].agg(['mean', 'count']).reset_index()
venue_stats.columns = ['Venue', 'venue_avg_score', 'venue_races']

final_df = df.merge(historical_stats, on='Sailor', how='left')
final_df = final_df.merge(partner_stats, on='Partner', how='left')
final_df = final_df.merge(venue_stats, on='Venue', how='left')

atts = [
    'Div_encoded', 
    'Position_encoded',
    'avg_score', 
    'best_score', 
    'worst_score', 
    'races_completed',
    'partner_avg_score',
    'partner_races',
    'venue_avg_score',
    'venue_races'
]

X = final_df[atts]
y = final_df['Score']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = RandomForestRegressor(n_estimators=100, n_jobs=1, random_state=42)
model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_test_scaled)

In [48]:
# Calculate the MSE and r2 for the model
print("\nModel Performance:")
print(f"Mean Squared Error: {mean_squared_error(y_test, y_pred):.2f}")
print(f"R² Score: {r2_score(y_test, y_pred):.2f}")


Model Performance:
Mean Squared Error: 13.48
R² Score: 0.43


In [51]:
# Inspect which features the model relied on most
feature_cols = atts

feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

print("\nFeature Importance:")
print(feature_importance)


Feature Importance:
             feature  importance
2          avg_score    0.362935
8    venue_avg_score    0.156682
6  partner_avg_score    0.150015
5    races_completed    0.086414
7      partner_races    0.079919
9        venue_races    0.062898
4        worst_score    0.043754
0        Div_encoded    0.026601
3         best_score    0.017460
1   Position_encoded    0.013323


## Race-Level Tests

To sanity-check the model, I run predictions on a few specific races (`raceID`) and compare:
- actual `Score`
- predicted `Score`
- MAE (average absolute difference in finishing position)

In [54]:
# Race test: MCSA Fall Open (1A)
race = "f24/mcsa-open-fall/1A"
race_df = final_df[final_df['raceID'] == race]

X_race = race_df[atts]
X_race_scaled = scaler.transform(X_race)


race_predictions = model.predict(X_race_scaled)

# making a new df that includes the sailor's name, actual score, and predicted score
comparison_df = race_df[['Sailor', 'Team', 'Score']].copy()
comparison_df['Predicted Score'] = race_predictions
comparison_df.rename(columns={'Score': 'Actual Score'}, inplace=True)


print(comparison_df)
# mean absolute error
mae = mean_absolute_error(comparison_df['Actual Score'], comparison_df['Predicted Score'])
print(f"Mean Absolute Error (MAE) for predictions in Race {race}: {mae:.2f}")

                   Sailor                Team  Actual Score  Predicted Score
1821       Jake Weinstein        Northwestern             8         3.935542
1822       Herbert Single        Northwestern             8         4.492128
1861         Braden Vogel            Michigan             5         4.116206
1862        Nina Gonzalez            Michigan             5         4.319341
1901        Timothy Hesse          Notre Dame             6         3.753762
1902        Aidan Kiergan          Notre Dame             6         3.472963
1941       Andrew Michels       Michigan Tech             7         5.145191
1942      Lucas Rodenroth       Michigan Tech             7         8.369556
1981         Eric Brieden           Marquette             4         6.736050
1982         Colin Hexter           Marquette             4         6.215801
2021   Nithya Balachander             Indiana             1         5.288532
2022  Colin Herbolsheimer             Indiana             1         4.811263

In [56]:
# Race test: Lake Virginia Invitational (1A)
race = "f24/lake-virginia-invitational/1A"
race_df = final_df[final_df['raceID'] == race]

X_race = race_df[atts]
X_race_scaled = scaler.transform(X_race)


race_predictions = model.predict(X_race_scaled)


comparison_df = race_df[['Sailor', 'Team', 'Score']].copy()
comparison_df['Predicted Score'] = race_predictions
comparison_df.rename(columns={'Score': 'Actual Score'}, inplace=True)


print(comparison_df)
mae = mean_absolute_error(comparison_df['Actual Score'], comparison_df['Predicted Score'])

print(f"Mean Absolute Error (MAE) for predictions in Race {race}: {mae:.2f}")

                    Sailor           Team  Actual Score  Predicted Score
17306         Cole Schweda   Jacksonville             3         3.182813
17307         Kayla Putzke   Jacksonville             3         2.221769
17352      Brent Penwarden   Jacksonville             2         4.057267
17353         Rori Bywater   Jacksonville             2         3.698007
17400     Emma Shakespeare  South Florida             1         2.633378
17401        Madisen Hamai  South Florida             1         2.345112
17440     Humberto Porrata  South Florida             8         5.910791
17441         Ruth Bergman  South Florida             8         4.808193
17488           David Webb       U. Miami            11         7.617119
17489      Isaac Van Buren       U. Miami            11         8.713566
17536          Oliver West       U. Miami            12         9.531118
17537        Jospeh Iacono       U. Miami            12         9.435891
17584          Blake March  South Florida          

In [58]:
# Race test: MCSA Fall Open (10A)
race = "f24/mcsa-open-fall/10A"
race_df = final_df[final_df['raceID'] == race]

X_race = race_df[atts]
X_race_scaled = scaler.transform(X_race)


race_predictions = model.predict(X_race_scaled)


comparison_df = race_df[['Sailor', 'Team', 'Score']].copy()
comparison_df['Predicted Score'] = race_predictions
comparison_df.rename(columns={'Score': 'Actual Score'}, inplace=True)


print(comparison_df)
mae = mean_absolute_error(comparison_df['Actual Score'], comparison_df['Predicted Score'])


print(f"Mean Absolute Error (MAE) for predictions in Race {race}: {mae:.2f}")

                   Sailor                Team  Actual Score  Predicted Score
1839       Jake Weinstein        Northwestern             7         3.935542
1840       Herbert Single        Northwestern             7         4.492128
1874         Braden Vogel            Michigan             1         3.050806
1880           Alden Gort            Michigan             1         1.324000
1919        Timothy Hesse          Notre Dame             3         3.753762
1920        Aidan Kiergan          Notre Dame             3         3.472963
1952       Andrew Michels       Michigan Tech             2         7.138277
1960        Gavin Parsons       Michigan Tech             2         5.567183
1999         Eric Brieden           Marquette             4         6.736050
2000         Colin Hexter           Marquette             4         6.215801
2039   Nithya Balachander             Indiana             8         5.288532
2040  Colin Herbolsheimer             Indiana             8         4.811263

## Interpretation

- Holdout performance: **MSE ≈ 13.47**, **R² ≈ 0.43**
- Race-level MAE across examples: **~1.6–2.3 places**
- Most influential features:
  - historical average finish
  - partner aggregate performance
  - venue aggregate performance

Model accuracy is limited by unobserved race conditions such as wind and weather.

## Potential Improvements

- Add weather and wind conditions
- Include fleet size and event competitiveness
- Incorporate sailor experience indicators (years racing, program strength)

Sailing outcomes are inherently noisy, but richer context features should improve generalization.